# 🎓 Student Success Predictor — End-to-End ML Capstone

## Objective
Predict `Exam_Score` using student academic, learning, family, and school-related factors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/student_performance.csv")
df.shape, df.head()

## 1. Data Quality

In [ ]:
print(df.info())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(10))
print("Duplicates:", df.duplicated().sum())

## 2. Exploratory Data Analysis

In [ ]:
display(df.describe(include="all").T)
plt.figure(figsize=(8,5))
sns.histplot(df["Exam_Score"], kde=True)
plt.title("Exam Score Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Hours_Studied", y="Exam_Score", alpha=.4)
plt.title("Study Hours vs Exam Score")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Attendance", y="Exam_Score", alpha=.4)
plt.title("Attendance vs Exam Score")
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
def engineer(d):
    d=d.copy()
    d["Study_Attendance_Index"]=d["Hours_Studied"]*d["Attendance"]/100
    d["Academic_Consistency"]=(d["Previous_Scores"]+d["Attendance"])/2
    d["Study_Sleep_Index"]=d["Hours_Studied"]*d["Sleep_Hours"]
    return d

X = engineer(df.drop(columns="Exam_Score"))
y = df["Exam_Score"]

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

cat = X.select_dtypes(include="object").columns.tolist()
num = X.select_dtypes(exclude="object").columns.tolist()

pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat)
])
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42)

## 4. Train Multiple Models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10),
    "Random Forest": RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=.04, max_depth=3, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=250, random_state=42, n_jobs=-1)
}

results=[]
pipelines={}
for name, model in models.items():
    pipe=Pipeline([("preprocessor",pre),("model",model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    results.append([name,
                    mean_absolute_error(y_test,pred),
                    mean_squared_error(y_test,pred)**0.5,
                    r2_score(y_test,pred)])
    pipelines[name]=pipe

results_df=pd.DataFrame(results,columns=["Model","MAE","RMSE","R2"]).sort_values("R2",ascending=False)
display(results_df)

## 5. Select & Save Best Model

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = pipelines[best_name]
print("Selected:", best_name)
import joblib
joblib.dump(best_model, "../model.pkl")
print("Saved to ../model.pkl")

## 6. Conclusion
The selected model is deployed through Streamlit. Always validate the model on independent real-world data before high-impact use.